# Figure 5

In [1]:
import numpy as np
import pickle
from flax import nnx
from pathlib import Path
import string
from jax import numpy as jnp
import jax
import sys

# plot related
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.legend_handler import HandlerTuple

plt.rcParams["svg.fonttype"] = "none"

# model related
from nntp.utils.datatype import mean_batch_metrics, transpose_metric_history
from nntp.utils.model import restore_model_from_checkpoint
from nntp.utils.plot import (
    reorder_legend_handles_row_major,
    find_experiments,
    load_experiments_parallel,
)

from common import (
    subtitle_fontsize,
    panel_indexing_fontsize,
    feature_size,
    output_size,
    CHANNEL,
    CHANNEL_NAME_MAPPING,
    COLORS,
)

src_path = (Path.cwd().parent / "src").resolve()
sys.path.insert(0, str(src_path))
from CoSynRNNModel.CoSynRNNModel import CoSynRNN

In [2]:
def get_experiment_tasks(name):
    if name == "maximum":
        return [
            "dm2-ry",
            "delayanti-ry",
            "delaydm2-ry",
            "dm1-ry",
            "reactanti-ry",
            "contextdelaydm1-ry",
            "multidm-ry",
            "reactgo-ry",
            "multidelaydm-ry",
            "delaygo-ry",
            "delaydm1-ry",
            "fdanti-ry",
            "contextdm1-ry",
            "contextdelaydm2-ry",
            "fdgo-ry",
            "contextdm2-ry",
            "free",
        ]
    elif name == "minimum":
        return [
            "reactanti-ry",
            "reactgo-ry",
            "dm2-ry",
            "contextdm2-ry",
            "multidm-ry",
            "contextdm1-ry",
            "dm1-ry",
            "contextdelaydm1-ry",
            "delaydm1-ry",
            "multidelaydm-ry",
            "contextdelaydm2-ry",
            "delaydm2-ry",
            "delaygo-ry",
            "delayanti-ry",
            "fdanti-ry",
            "fdgo-ry",
            "free",
        ]


# create path
out_path = Path("./output")
out_path.mkdir(parents=True, exist_ok=True)
root_path = Path("..").expanduser().resolve() / "figures/full"
experiments = find_experiments(root_path, get_experiment_tasks)
for path in experiments:
    print(path)


def load_experiment(_, path):
    config = pickle.load(open(path / "metadata_config.blob", "rb"))
    model = restore_model_from_checkpoint(
        path,
        CoSynRNN(nnx.Rngs(42), feature_size, output_size, config),
        1600,
    )

    train_metric_history = pickle.load(
        open(path / "metadata_train_mean_metrics.blob", "rb")
    )
    validation_metric_history = pickle.load(
        open(path / "metadata_validation_mean_metrics.blob", "rb")
    )
    # ================= extra neurons =================
    W_rec_active_synapse = jnp.abs(model.W_rec) > model.threshold

    recurrent_free_neurons = (
        (W_rec_active_synapse.sum(axis=0) == 0)
        & (W_rec_active_synapse.sum(axis=1) == 0)
        & model.trainable_mask
    )

    cue = int(jax.device_get(model.cue.get_value()))

    readout_free_neurons = (
        jnp.sum(
            jnp.abs(model.W_out * model.modulation_activation(model.W_mask[cue]))
            > model.threshold,
            axis=-1,
        )
        == 0
    )
    trainable_mask_value = recurrent_free_neurons & readout_free_neurons

    task_index = jnp.max(model.identity)
    identity_value = jnp.where(trainable_mask_value, task_index + 1, model.identity)
    model.identity = identity_value

    return {
        "model": model,
        "train": train_metric_history,
        "validation": validation_metric_history,
    }


loaded_experiments = load_experiments_parallel(experiments, load_experiment)

{'tasks': ['dm2-ry', 'delayanti-ry', 'delaydm2-ry', 'dm1-ry', 'reactanti-ry', 'contextdelaydm1-ry', 'multidm-ry', 'reactgo-ry', 'multidelaydm-ry', 'delaygo-ry', 'delaydm1-ry', 'fdanti-ry', 'contextdm1-ry', 'contextdelaydm2-ry', 'fdgo-ry', 'contextdm2-ry', 'free'], 'paths': [PosixPath('/allen/aind/scratch/ivan.y.gao/CoSyn-RNN/figures/full/maximum/25156048/1/f0aaf813bc302975'), PosixPath('/allen/aind/scratch/ivan.y.gao/CoSyn-RNN/figures/full/maximum/25156048/42/60a59487df1add0e'), PosixPath('/allen/aind/scratch/ivan.y.gao/CoSyn-RNN/figures/full/maximum/25156048/128/e74498f3c9316246'), PosixPath('/allen/aind/scratch/ivan.y.gao/CoSyn-RNN/figures/full/maximum/25156048/128128/3cfaa882659484a4')]}
{'tasks': ['reactanti-ry', 'reactgo-ry', 'dm2-ry', 'contextdm2-ry', 'multidm-ry', 'contextdm1-ry', 'dm1-ry', 'contextdelaydm1-ry', 'delaydm1-ry', 'multidelaydm-ry', 'contextdelaydm2-ry', 'delaydm2-ry', 'delaygo-ry', 'delayanti-ry', 'fdanti-ry', 'fdgo-ry', 'free'], 'paths': [PosixPath('/allen/aind/sc

In [3]:
# from Figure2.ipynb


def plot_experiments(
    ax,
    experiments,
    metric,
    title,
    logY=False,
    show_xlabel=False,
    show_ylabel=False,
    metric_key="train",
):
    tasks = experiments["tasks"]
    data = experiments["data"]

    if metric == "sparsity":
        curves = np.stack(
            [
                np.asarray(
                    seed_data[metric_key]["model"]["loss/sparsity"],
                    dtype=float,
                )
                for seed_data in data
            ],
            axis=0,
        )

        mean_curve = np.nanmean(curves, axis=0)
        lower = np.nanmin(curves, axis=0)
        upper = np.nanmax(curves, axis=0)
        x = np.arange(mean_curve.size)

        ax.fill_between(
            x,
            lower,
            upper,
            color="gray",
            alpha=0.4,
            linewidth=0,
        )

        ax.plot(
            x,
            mean_curve,
            color="gray",
            linewidth=1,
        )

    elif metric == "amplification":
        curves = np.stack(
            [
                np.asarray(
                    seed_data[metric_key]["model"]["penalty_amplification"],
                    dtype=float,
                )
                for seed_data in data
            ],
            axis=0,
        )

        mean_curve = np.nanmean(curves, axis=0)
        lower = np.nanmin(curves, axis=0)
        upper = np.nanmax(curves, axis=0)
        x = np.arange(mean_curve.size)

        ax.fill_between(
            x,
            lower,
            upper,
            color="gray",
            alpha=0.4,
            linewidth=0,
        )

        ax.plot(
            x,
            mean_curve,
            color="gray",
            linewidth=1,
        )

        ax.axhline(
            y=1.0,
            xmin=0.05,
            xmax=0.95,
            color="black",
            linestyle="--",
            linewidth=0.8,
            alpha=0.7,
            zorder=1,
        )

    else:
        plot_tasks = [task for task in tasks if task != "free"]

        for task in plot_tasks:
            if metric == "train":
                curves = np.stack(
                    [
                        np.asarray(
                            seed_data[metric_key]["task"]["loss"][task]
                            - seed_data[metric_key]["model"]["loss/sparsity"],
                            dtype=float,
                        )
                        for seed_data in data
                    ],
                    axis=0,
                )

            elif metric == "validation":
                curves = np.stack(
                    [
                        np.asarray(
                            seed_data[metric_key]["task"]["loss"][task],
                            dtype=float,
                        )
                        for seed_data in data
                    ],
                    axis=0,
                )

            elif metric == "accuracy":
                curves = np.stack(
                    [
                        np.asarray(
                            seed_data[metric_key]["task"]["accuracy_angle"][task],
                            dtype=float,
                        )
                        for seed_data in data
                    ],
                    axis=0,
                )

            else:
                raise ValueError(f"Unsupported metric: {metric}")

            mean_curve = np.nanmean(curves, axis=0)
            lower = np.nanmin(curves, axis=0)
            upper = np.nanmax(curves, axis=0)
            x = np.arange(mean_curve.size)

            color = COLORS[task]

            ax.fill_between(
                x,
                lower,
                upper,
                color=color,
                alpha=0.4,
                linewidth=0,
            )

            ax.plot(
                x,
                mean_curve,
                color=color,
                linewidth=1,
                label=task.removesuffix("-ry"),
            )

    if metric == "accuracy":
        chance = (2 * 36) / 360

        ax.axhline(
            y=chance,
            xmin=0.05,
            xmax=0.95,
            color="black",
            linestyle="--",
            linewidth=0.8,
            alpha=0.7,
            zorder=1,
        )

        ax.text(
            0.94,
            chance - 0.2,
            "Chance",
            transform=ax.get_yaxis_transform(),
            ha="right",
            va="top",
            fontsize=10,
            color="black",
        )

        ax.set_ylim(0, 1)
        ax.set_yticks([0, 1])
        ax.tick_params(
            axis="y",
            which="major",
            length=0,
        )

    if logY:
        ax.set_yscale("log")

    if show_xlabel:
        ax.set_xlabel("Epoch")
        ax.tick_params(
            axis="x",
            which="both",
            bottom=True,
            labelbottom=True,
        )
    else:
        ax.tick_params(
            axis="x",
            which="both",
            bottom=False,
            labelbottom=False,
        )

    if show_ylabel:
        if metric in {"train", "validation", "sparsity"}:
            ylabel = "Loss"
        elif metric == "accuracy":
            ylabel = "Accuracy"
        else:
            ylabel = metric.replace("_", " ").capitalize()

        if logY:
            ylabel += " (log)"

        ax.set_ylabel(ylabel)
        ax.tick_params(
            axis="y",
            which="both",
            left=True,
            labelleft=True,
        )
        ax.tick_params(
            axis="y",
            which="minor",
            left=False,
            labelleft=False,
        )
    else:
        ax.set_ylabel("")
        ax.tick_params(
            axis="y",
            which="both",
            left=False,
            labelleft=False,
        )

    ax.set_title(title)

    sns.despine(
        ax=ax,
        top=True,
        right=True,
        left=not show_ylabel,
        bottom=not show_xlabel,
    )

    ax.grid(
        axis="x",
        which="major",
        linestyle="--",
        linewidth=0.6,
        alpha=0.5,
    )

    return ax

In [4]:
# from Figure3.ipynb
def plot_model_recurrent(ax, tasks, W_value, identity_value, threshold, show_cb):
    W_value = np.asarray(W_value)
    identity_value = np.asarray(identity_value).astype(int)

    active_mask = np.abs(W_value) > threshold

    # identity stores ordered task IDs: 0, 1, 2, ...
    task_ids = np.unique(identity_value)
    task_ids = task_ids[task_ids >= 0]

    orders = []
    block_labels = []

    for task_id in task_ids:
        task_indices = np.where(identity_value == task_id)[0]

        if len(task_indices) == 0:
            continue

        # Active connections within this block
        block_active = active_mask[np.ix_(task_indices, task_indices)]

        # Sort neurons by active connection count
        block_row_count = active_mask[task_indices, :].sum(axis=1)
        block_col_count = block_active.sum(axis=0)
        block_score = block_row_count + block_col_count

        task_order = task_indices[np.argsort(-block_score)]

        orders.append(task_order)
        block_labels.append(task_id)

    if not orders:
        raise ValueError("No valid task IDs were found in identity_value.")

    order = np.concatenate(orders)

    W_sorted = W_value[np.ix_(order, order)]
    W_display = np.clip(W_sorted, -threshold, threshold)

    # ==============================================
    # Heatmap
    # ==============================================
    image = ax.imshow(
        W_display,
        aspect="equal",
        cmap="coolwarm",
        norm=TwoSlopeNorm(
            vmin=-threshold,
            vcenter=0.0,
            vmax=threshold,
        ),
        interpolation="nearest",
    )
    # ==============================================
    # Block information
    # ==============================================
    block_sizes = np.asarray([len(task_order) for task_order in orders], dtype=int)
    block_starts = np.concatenate(
        [
            [0],
            np.cumsum(block_sizes)[:-1],
        ]
    )

    boundaries = np.cumsum(block_sizes)[:-1]

    # ==============================================
    # Block boundaries on the heatmap
    # ==============================================
    for boundary in boundaries:
        ax.axhline(boundary - 0.5, color="black", linewidth=0.5)
        ax.axvline(boundary - 0.5, color="black", linewidth=0.5)

    # ==============================================
    # Colors for different identity blocks
    # ==============================================
    # Full task names, used to look up colors
    block_task_names = [tasks[int(task_id)] for task_id in block_labels]

    # Use your predefined task colors
    block_colors = [COLORS[task_name] for task_name in block_task_names]
    num_blocks = len(block_labels)
    block_cmap = ListedColormap(block_colors)

    # Expand block index to one value per sorted neuron
    block_index_per_neuron = np.empty(
        len(order),
        dtype=int,
    )

    for block_index, (start, size) in enumerate(zip(block_starts, block_sizes)):
        block_index_per_neuron[start : start + size] = block_index

    # ==============================================
    # Top horizontal block strip
    # ==============================================
    ax_top_strip = ax.inset_axes(
        [
            0.0,  # x position
            1.01,  # y position
            1.0,  # width
            0.02,  # height
        ]
    )

    ax_top_strip.imshow(
        block_index_per_neuron[np.newaxis, :],
        aspect="auto",
        cmap=block_cmap,
        vmin=-0.5,
        vmax=num_blocks - 0.5,
        interpolation="nearest",
    )

    ax_top_strip.set_xticks([])
    ax_top_strip.set_yticks([])

    for spine in ax_top_strip.spines.values():
        spine.set_visible(False)

    # ==============================================
    # Left vertical task block strip
    # ==============================================
    ax_left_strip = ax.inset_axes(
        [
            -0.03,  # x position
            0.0,  # y position
            0.02,  # width
            1.0,  # height
        ]
    )

    ax_left_strip.imshow(
        block_index_per_neuron[:, np.newaxis],
        aspect="auto",
        cmap=block_cmap,
        vmin=-0.5,
        vmax=num_blocks - 0.5,
        interpolation="nearest",
    )

    ax_left_strip.set_xticks([])
    ax_left_strip.set_yticks([])

    for spine in ax_left_strip.spines.values():
        spine.set_visible(False)

    # ==============================================
    # Weight colorbar
    # ==============================================
    if show_cb:
        ax_cbar = ax.inset_axes(
            [
                1.010,  # x position
                0.0,  # y position
                0.025,  # width
                1.0,  # height
            ]
        )
        cbar = ax.figure.colorbar(image, cax=ax_cbar)
        cbar.set_label(r"Weight ($\times 10^{-4}$)", labelpad=-10)
        cbar.set_ticks([-threshold, threshold])
        cbar.ax.yaxis.set_major_formatter(
            FuncFormatter(lambda value, position: f"{value / 1e-4:g}")
        )
        cbar.outline.set_visible(False)

    # ==============================================
    # Labels
    # ==============================================
    ax.set_xlabel("Presynaptic Neurons")
    if not show_cb:
        ax.set_ylabel("Postsynaptic Neurons", labelpad=22)
    ax.tick_params(
        axis="y",
        which="both",
        left=False,
        labelleft=False,
    )
    return order

In [5]:
def plot_experiment(
    axes_top, ax_bottom, tasks, model, experiments, show_ylabel, show_cb
):
    plot_experiments(
        axes_top[0],
        experiments,
        metric="train",
        title="Training Task Loss",
        logY=True,
        show_ylabel=show_ylabel,
    )
    plot_experiments(
        axes_top[1],
        experiments,
        metric="accuracy",
        title="Training Accuracy",
        show_ylabel=show_ylabel,
    )
    plot_experiments(
        axes_top[2],
        experiments,
        metric="validation",
        title="Validation Loss",
        logY=True,
        show_ylabel=show_ylabel,
        metric_key="validation",
    )
    plot_experiments(
        axes_top[3],
        experiments,
        metric="accuracy",
        title="Validation Accuracy",
        show_ylabel=show_ylabel,
        metric_key="validation",
    )
    plot_experiments(
        axes_top[4],
        experiments,
        metric="sparsity",
        title="Training Sparsity Loss",
        show_ylabel=show_ylabel,
    )
    plot_experiments(
        axes_top[5],
        experiments,
        metric="amplification",
        title="Loss-Dependent Amplification Factor",
        show_xlabel=True,
        show_ylabel=show_ylabel,
    )

    threshold = model.threshold
    W_out = model.W_out

    plot_model_recurrent(
        ax_bottom,
        tasks,
        model.W_rec.T,
        model.identity,
        threshold,
        show_cb=show_cb,
    )

In [6]:
for index, seed in enumerate([1, 42, 128, 128128]):
    # for index, seed in enumerate([1]):
    fig = plt.figure(figsize=(17, 17))

    # 外层：上方曲线组 + 下方 heatmap
    outer_gs = fig.add_gridspec(
        nrows=2,
        ncols=2,
        width_ratios=[1, 1],
        height_ratios=[5, 6],
        hspace=-0.02,  # 只控制曲线组与 heatmap 的间距
        wspace=0.08,
    )

    # ==============================================
    # 上面 5 行小图
    # ==============================================
    top_left_gs = outer_gs[0, 0].subgridspec(
        nrows=6,
        ncols=1,
        hspace=0.28,  # 只控制左侧五个小图之间的间距
    )

    top_right_gs = outer_gs[0, 1].subgridspec(
        nrows=6,
        ncols=1,
        hspace=0.28,  # 只控制右侧五个小图之间的间距
    )

    axes_top_left = []
    axes_top_right = []

    for row in range(6):
        if row == 0:
            ax_left = fig.add_subplot(top_left_gs[row, 0])
        else:
            ax_left = fig.add_subplot(
                top_left_gs[row, 0],
                sharex=axes_top_left[0],
            )

        ax_right = fig.add_subplot(
            top_right_gs[row, 0],
            sharex=ax_left,
            sharey=ax_left,
        )

        axes_top_left.append(ax_left)
        axes_top_right.append(ax_right)

    # ==============================================
    # 下面 1 行大图
    # ==============================================
    ax_bottom_left = fig.add_subplot(outer_gs[1, 0])

    ax_bottom_right = fig.add_subplot(
        outer_gs[1, 1],
        sharex=ax_bottom_left,
        sharey=ax_bottom_left,
    )

    plot_experiment(
        axes_top_left,
        ax_bottom_left,
        loaded_experiments[1]["tasks"],
        loaded_experiments[1]["data"][index]["model"],
        loaded_experiments[1],
        show_ylabel=True,
        show_cb=False,
    )

    plot_experiment(
        axes_top_right,
        ax_bottom_right,
        loaded_experiments[0]["tasks"],
        loaded_experiments[0]["data"][index]["model"],
        loaded_experiments[0],
        show_ylabel=False,
        show_cb=True,
    )

    fig.text(
        0.30,
        0.97,
        "Minimum-Cost Order",
        ha="center",
        va="center",
        fontsize=subtitle_fontsize,
        fontweight="bold",
    )

    fig.text(
        0.70,
        0.97,
        "Maximum-Cost Order",
        ha="center",
        va="center",
        fontsize=subtitle_fontsize,
        fontweight="bold",
    )

    # ==============================================
    # Legend
    # ==============================================
    def make_task_legend_items(tasks):
        items = []

        for task in tasks:
            display_name = task.removesuffix("-ry")

            if display_name.lower() in {"free", "available"}:
                continue

            color = COLORS[task]

            line = Line2D(
                [0],
                [0],
                color=color,
                linewidth=2,
            )

            patch = Patch(
                facecolor=color,
                edgecolor="none",
            )

            items.append(
                (
                    (patch, line),
                    CHANNEL_NAME_MAPPING[display_name],
                )
            )

        return items

    def reorder_legend_items_row_major(items, ncol):
        nrow = int(np.ceil(len(items) / ncol))

        return [
            items[row * ncol + col]
            for col in range(ncol)
            for row in range(nrow)
            if row * ncol + col < len(items)
        ]

    forward_items = reorder_legend_items_row_major(
        make_task_legend_items(loaded_experiments[1]["tasks"]),
        ncol=4,
    )

    forward_handles, forward_labels = zip(*forward_items)

    fig.legend(
        handles=forward_handles,
        labels=forward_labels,
        handler_map={
            tuple: HandlerTuple(
                ndivide=None,
                pad=0.1,
            ),
        },
        loc="upper center",
        bbox_to_anchor=(0.31, 0.96),
        ncol=4,
        frameon=False,
    )

    reverse_items = reorder_legend_items_row_major(
        make_task_legend_items(loaded_experiments[0]["tasks"]),
        ncol=4,
    )

    reverse_handles, reverse_labels = zip(*reverse_items)

    fig.legend(
        handles=reverse_handles,
        labels=reverse_labels,
        handler_map={
            tuple: HandlerTuple(
                ndivide=None,
                pad=0.1,
            ),
        },
        loc="upper center",
        bbox_to_anchor=(0.72, 0.96),
        ncol=4,
        frameon=False,
    )

    fig.subplots_adjust(
        top=0.88,
        bottom=0.04,
    )
    fig.align_ylabels(axes_top_left)

    panel_axes = []

    # 上面 5 行，按每一行左、右排列
    for ax_left, ax_right in zip(axes_top_left, axes_top_right):
        panel_axes.extend([ax_left, ax_right])

    # 最下面两张 heatmap
    panel_axes.extend([ax_bottom_left, ax_bottom_right])

    for label, ax in zip(string.ascii_lowercase, panel_axes):
        ax.text(
            0.01,
            1.05,
            f"{label}",
            transform=ax.transAxes,
            ha="left",
            va="bottom",
            fontsize=panel_indexing_fontsize,
            fontweight="bold",
            clip_on=False,
        )

    filename_png = (
        "Figure5.png" if seed == 1 else f"SupplementaryFigure4_seed_{seed}.png"
    )
    # filename_svg = (
    #     "Figure5.svg" if seed == 1 else f"SupplementaryFigure3_seed_{seed}.svg"
    # )

    fig.savefig(
        out_path / filename_png,
        dpi=300,
        bbox_inches="tight",
    )
    # fig.savefig(
    #     out_path / filename_svg,
    #     bbox_inches="tight",
    # )
    plt.close(fig)

/tmp/ipykernel_303260/2839898893.py:138: RuntimeWarning: Mean of empty slice
  mean_curve = np.nanmean(curves, axis=0)
/tmp/ipykernel_303260/2839898893.py:139: RuntimeWarning: All-NaN slice encountered
  lower = np.nanmin(curves, axis=0)
/tmp/ipykernel_303260/2839898893.py:140: RuntimeWarning: All-NaN slice encountered
  upper = np.nanmax(curves, axis=0)
